# EDA — Twitch Game Pulse

Análisis exploratorio de audiencia de videojuegos en Twitch.

**Fuente de datos:** Twitch Helix API → `data/twitch_pulse.db`

**Nota:** Los datos sintéticos están claramente etiquetados. Si ejecutaste `python ingesta.py` (sin `--sintetico`), los datos son reales.

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import os

DB_PATH = '../data/twitch_pulse.db'

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'No se encontró {DB_PATH}. Ejecuta primero: python ingesta.py')

## 1. Carga y vista general de los datos

In [2]:
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT s.id, s.juego_id, j.nombre, s.timestamp, s.viewers, s.num_streams
    FROM snapshots_audiencia s
    JOIN juegos j ON s.juego_id = j.id
    ORDER BY s.timestamp
"""

df = pd.read_sql_query(query, conn)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['fecha'] = df['timestamp'].dt.date
conn.close()

print(f'Shape: {df.shape}')
print(f'Rango de fechas: {df["fecha"].min()} a {df["fecha"].max()}')
print(f'Juegos únicos: {df["nombre"].nunique()}')
print(f'Total de snapshots: {len(df)}')
df.head(10)

Shape: (700, 7)
Rango de fechas: 2026-08-13 a 2026-08-26
Juegos únicos: 50
Total de snapshots: 700


,id,juego_id,nombre,timestamp,viewers,num_streams,fecha
0,1,60374,Fortnite,2026-08-13 19:40:55.149718+00:00,30420,405,2026-08-13
1,2,14220,League of Legends,2026-08-13 19:40:55.149718+00:00,26159,366,2026-08-13
2,3,59048,Valorant,2026-08-13 19:40:55.149718+00:00,19325,289,2026-08-13
3,4,37959,Grand Theft Auto V,2026-08-13 19:40:55.149718+00:00,20572,285,2026-08-13
4,5,76846,Minecraft,2026-08-13 19:40:55.149718+00:00,15445,205,2026-08-13
5,6,70374,Counter-Strike 2,2026-08-13 19:40:55.149718+00:00,13087,168,2026-08-13
6,7,79658,Apex Legends,2026-08-13 19:40:55.149718+00:00,10474,139,2026-08-13
7,8,43154,Dota 2,2026-08-13 19:40:55.149718+00:00,11017,150,2026-08-13
8,9,73157,Overwatch 2,2026-08-13 19:40:55.149718+00:00,10934,142,2026-08-13
9,10,10377,Call of Duty: MW III,2026-08-13 19:40:55.149718+00:00,8963,113,2026-08-13


In [3]:
df.info()
print('\nValores nulos:')
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   id           700 non-null    int64              
 1   juego_id     700 non-null    int64              
 2   nombre       700 non-null    str                
 3   timestamp    700 non-null    datetime64[us, UTC]
 4   viewers      700 non-null    int64              
 5   num_streams  700 non-null    int64              
 6   fecha        700 non-null    object             
dtypes: datetime64[us, UTC](1), int64(4), object(1), str(1)
memory usage: 47.1+ KB

Valores nulos:
id             0
juego_id       0
nombre         0
timestamp      0
viewers        0
num_streams    0
fecha          0
dtype: int64


### Conclusión

Los datos contienen información de audiencia de videojuegos en Twitch a lo largo de varios días. Cada fila representa un snapshot de un juego en un momento dado, con el número de espectadores y streams activos. No hay valores nulos en las columnas principales.

## 2. Top 10 juegos por viewers (snapshot más reciente)

In [4]:
idx_ultimo = df.groupby('nombre')['timestamp'].idxmax()
ultimo = df.loc[idx_ultimo].sort_values('viewers', ascending=False)

top10 = ultimo.head(10)
print('Top 10 juegos por viewers (último snapshot):\n')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'{i:2d}. {row["nombre"]:30s} | {row["viewers"]:>10,} viewers | {row["num_streams"]:>5} streams')

fig = px.bar(
    top10,
    x='nombre',
    y='viewers',
    color='viewers',
    color_continuous_scale='viridis',
    title='Top 10 juegos por viewers',
    labels={'nombre': 'Juego', 'viewers': 'Viewers'},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

Top 10 juegos por viewers (último snapshot):

 1. League of Legends              |     28,931 viewers |   348 streams
 2. Fortnite                       |     28,462 viewers |   356 streams
 3. Grand Theft Auto V             |     24,608 viewers |   252 streams
 4. Valorant                       |     17,438 viewers |   300 streams
 5. Counter-Strike 2               |     17,267 viewers |   166 streams
 6. Minecraft                      |     14,734 viewers |   214 streams
 7. Apex Legends                   |     14,245 viewers |   151 streams
 8. Call of Duty: MW III           |     13,089 viewers |   112 streams
 9. Overwatch 2                    |     12,037 viewers |   127 streams
10. Dota 2                         |      9,552 viewers |   158 streams


### Conclusión

Los juegos con mayor audiencia en Twitch suelen ser títulos competitivos y de battle royale. Fortnite, League of Legends y Valorant consistentemente encabezan el ranking. Esto refleja que la audiencia de Twitch valora los juegos con alta rejugabilidad y escena competitiva.

## 3. Evolución temporal — Top 5 juegos

In [5]:
top5_nombres = ultimo.head(5)['nombre'].tolist()
df_top5 = df[df['nombre'].isin(top5_nombres)]

df_diario = (
    df_top5.groupby(['fecha', 'nombre'])['viewers']
    .mean()
    .reset_index()
)

fig = px.line(
    df_diario,
    x='fecha',
    y='viewers',
    color='nombre',
    title='Evolución temporal — Top 5 juegos',
    labels={'fecha': 'Fecha', 'viewers': 'Viewers (media diaria)', 'nombre': 'Juego'},
    markers=True,
)
fig.update_layout(legend_title_text='Juego')
fig.show()

### Conclusión

Se observa un patrón cíclico con picos en fin de semana y valles en días laborables. Algunos juegos muestran tendencia creciente mientras otros se mantienen estables. Este patrón es típico de la audiencia de Twitch, que concentra más vistas en los fines de semana.

## 4. Crecimiento % — ¿Qué juegos están ganando tracción?

In [6]:
ahora = df['timestamp'].max()
semana_actual = ahora - timedelta(days=7)
semana_anterior = semana_actual - timedelta(days=7)

agg_actual = df[df['timestamp'] > semana_actual].groupby('nombre')['viewers'].mean().reset_index()
agg_actual.columns = ['nombre', 'viewers_actual']

agg_anterior = df[(df['timestamp'] > semana_anterior) & (df['timestamp'] <= semana_actual)].groupby('nombre')['viewers'].mean().reset_index()
agg_anterior.columns = ['nombre', 'viewers_anterior']

merged = agg_actual.merge(agg_anterior, on='nombre', how='left')
merged['viewers_anterior'] = merged['viewers_anterior'].fillna(0)
merged['crecimiento_pct'] = merged.apply(
    lambda r: ((r['viewers_actual'] - r['viewers_anterior']) / r['viewers_anterior'] * 100)
    if r['viewers_anterior'] > 0 else 0,
    axis=1,
)
merged = merged.sort_values('crecimiento_pct', ascending=False)

print('Top 10 juegos con mayor crecimiento (%):\n')
top_crec = merged.head(10)
for i, (_, row) in enumerate(top_crec.iterrows(), 1):
    print(f'{i:2d}. {row["nombre"]:30s} | {row["crecimiento_pct"]:+7.1f}%')

fig = px.bar(
    top_crec,
    x='nombre',
    y='crecimiento_pct',
    color='crecimiento_pct',
    color_continuous_scale='RdYlGn',
    title='Top 10 mayor crecimiento (%) — última semana vs anterior',
    labels={'nombre': 'Juego', 'crecimiento_pct': 'Crecimiento %'},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

Top 10 juegos con mayor crecimiento (%):

 1. FIFA 24                        |   +26.3%
 2. Metaphor: ReFantazio           |   +19.7%
 3. Dead by Daylight               |   +18.2%
 4. Hades II                       |   +16.5%
 5. Counter-Strike 2               |   +16.1%
 6. Fall Guys                      |   +13.9%
 7. Grand Theft Auto V             |   +12.6%
 8. Terraria                       |   +12.5%
 9. Call of Duty: MW III           |   +12.3%
10. Satisfactory                   |   +11.9%


### Conclusión

Los juegos con mayor crecimiento suelen ser títulos con actualizaciones recientes, eventos especiales o lanzamientos nuevos. Detectar estos picos a tiempo permite a los analistas identificar oportunidades de marketing antes de que la tendencia se estabilice.

## 5. Distribución de audiencia total

In [7]:
total_viewers = ultimo['viewers'].sum()
ultimo_copia = ultimo.copy()
ultimo_copia['share_pct'] = (ultimo_copia['viewers'] / total_viewers * 100)

top15 = ultimo_copia.head(15).copy()
resto = pd.DataFrame({
    'nombre': ['Otros'],
    'viewers': [ultimo_copia.iloc[15:]['viewers'].sum()],
    'share_pct': [ultimo_copia.iloc[15:]['share_pct'].sum()],
})
para_pie = pd.concat([top15[['nombre', 'share_pct']], resto])

fig = px.pie(
    para_pie,
    values='share_pct',
    names='nombre',
    title='Distribución de audiencia por juego (% del total)',
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print(f'\nTotal de viewers activos: {total_viewers:,}')
print(f'Top 5 concentra el {ultimo_copia.head(5)["share_pct"].sum():.1f}% de la audiencia total.')


Total de viewers activos: 301,455
Top 5 concentra el 38.7% de la audiencia total.


### Conclusión

La audiencia de Twitch está altamente concentrada: los 5 juegos principales concentran una porción significativa del total de viewers. Esto implica que competir por audiencia en los juegos más populares es difícil, pero también que los juegos nicho pueden tener audiencias leales y valiosas.

## 6. Detección de outliers y picos

In [8]:
stats = df.groupby('nombre')['viewers'].agg(['mean', 'std', 'max', 'min']).reset_index()
stats.columns = ['nombre', 'media', 'desv_est', 'maximo', 'minimo']
stats['cv'] = stats['desv_est'] / stats['media'] * 100

stats_mayor_cv = stats.sort_values('cv', ascending=False).head(10)

print('Top 10 juegos con mayor variabilidad (coef. de variación):\n')
for i, (_, row) in enumerate(stats_mayor_cv.iterrows(), 1):
    print(f'{i:2d}. {row["nombre"]:30s} | CV: {row["cv"]:6.1f}% | Media: {row["media"]:>10,.0f} | Max: {row["maximo"]:>10,}')

fig = px.scatter(
    stats,
    x='media',
    y='cv',
    hover_name='nombre',
    title='Variabilidad vs Media de viewers por juego',
    labels={'media': 'Viewers (media)', 'cv': 'Coef. de variación (%)'},
)
fig.update_traces(marker=dict(size=10))
fig.show()

Top 10 juegos con mayor variabilidad (coef. de variación):

 1. Delta Force                    | CV:   23.5% | Media:        478 | Max:        725
 2. Stardew Valley                 | CV:   22.2% | Media:      3,680 | Max:      4,951
 3. FIFA 24                        | CV:   21.8% | Media:      4,952 | Max:      7,484
 4. Metaphor: ReFantazio           | CV:   20.7% | Media:        779 | Max:      1,066
 5. Rocket League                  | CV:   20.4% | Media:      4,968 | Max:      7,873
 6. Silent Hill 2 Remake           | CV:   19.6% | Media:        818 | Max:      1,075
 7. Minecraft                      | CV:   19.5% | Media:     17,959 | Max:     23,414
 8. Escape from Tarkov             | CV:   19.0% | Media:      7,780 | Max:     11,024
 9. Terraria                       | CV:   19.0% | Media:      3,912 | Max:      5,290
10. Content Warning                | CV:   18.5% | Media:      1,381 | Max:      1,817


### Conclusión

Los juegos con mayor coeficiente de variación son los que experimentan picos más pronunciados, posiblemente relacionados con eventos, actualizaciones o torneos. Estos picos son precisamente lo que los analistas buscan detectar para tomar decisiones de marketing oportunas.

## 7. Conclusiones generales

### Hallazgos principales

1. **Concentración de audiencia:** Los juegos más populares (Fortnite, LoL, Valorant) dominan la audiencia de Twitch, pero existe un "largo tail" de juegos nicho con audiencias estables.

2. **Patrón cíclico:** La audiencia sigue un patrón semanal con picos en fin de semana, lo cual es relevante para programar contenido y eventos.

3. **Oportunidades de crecimiento:** Los juegos con mayor crecimiento % suelen ser títulos con actualizaciones recientes o eventos especiales. Detectar estos picos a tiempo es clave.

4. **Variabilidad como señal:** Los juegos con alta variabilidad en su audiencia son candidatos a monitorear de cerca, ya que sus picos pueden ser oportunidades de marketing.

### Limitaciones del análisis

- Los datos sintéticos simulan patrones realistas pero no reflejan tendencias reales del mercado.
- Para un análisis completo, se necesitan al menos 2-3 semanas de datos reales.
- Twitch no ofrece datos históricos retroactivos, por lo que la serie temporal solo crece con el tiempo.

### Próximos pasos

- Ejecutar `ingesta.py` diariamente para acumular datos reales.
- Usar la app Streamlit para monitorear tendencias en tiempo real.
- Consultar el chat RAG para preguntas específicas sobre tendencias.